# Part 4: Feature Engineering

## Objectives
1. Apply log transformations to skewed variables
2. Calculate rolling volatility
3. Compute returns
4. Calculate real yields
5. Create momentum indicators

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

from src.data_loader import load_csv, save_csv
from src.feature_engineering import (
    apply_log_transformation,
    compute_rolling_volatility,
    compute_rolling_returns,
    compute_real_yield,
    compute_rolling_average
)

df = load_csv('../data/processed/merged_final_cleaned.csv')
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values('DATE')

## Log Transformations

In [ ]:
# Apply log transformation to skewed variables
skewed_cols = ['VIX', 'GPR_INDEX', 'FED_FUNDS_RATE']
df = apply_log_transformation(df, skewed_cols)

print(f"Shape after log transformation: {df.shape}")

## Volatility & Returns

In [ ]:
# Calculate returns
df = compute_rolling_returns(df, 'GOLD_PRICE')
df = compute_rolling_returns(df, 'SILVER_PRICE')
df = compute_rolling_returns(df, 'S&P_500')
df = compute_rolling_returns(df, 'WTI_CRUDE_OIL')
df = compute_rolling_returns(df, 'US_DOLLAR_INDEX')

# Calculate volatility (21-day rolling std)
df = compute_rolling_volatility(df, 'GOLD_PRICE', window=21)
df = compute_rolling_volatility(df, 'S&P_500', window=21)

## Economic Features

In [ ]:
# Calculate real yield
if '10Y_TREASURY_YIELD' in df.columns and 'CPI_INFLATION' in df.columns:
    df = compute_real_yield(df, '10Y_TREASURY_YIELD', 'CPI_INFLATION')

# Create moving averages
df = compute_rolling_average(df, 'VIX_log', window=5)

print(f"\nFinal shape: {df.shape}")
print(f"New columns created: {len(df.columns) - 22}")

## Save Engineered Features

In [ ]:
# Save with all features
save_csv(df, '../data/processed/merged_final_engineered.csv')

# Display sample
print("Sample of engineered features:")
feature_cols = [col for col in df.columns if '_log' in col or '_RET' in col or '_VOL' in col]
print(df[feature_cols].head())

print("\n✅ Feature engineering complete!")